# Compute the loss Hessian at a given epoch

Showcases `ntkpdf.hessian`: the **parameter-space Hessian** of the training loss the
fit minimised, per replica, at one epoch, plus its eigenvalues.

The loss is the diagonal-basis training chi2 normalised by the number of training
points,
$$L(\theta) = \tfrac{1}{N_{\rm tr}}\,(y_{\rm tr} - FK_{\rm tr}\,f(\theta))^{T}\,\mathrm{diag}(1/\Lambda_{\rm tr})\,(y_{\rm tr} - FK_{\rm tr}\,f(\theta)),$$
i.e. exactly the scalar built in `data_theory.loss_function_grid`. The network grid
$f(\theta)$ is differentiated through colibri's pure-JAX `grid_values_func`, so
`jax.hessian` gives the **full** $(P, P)$ Hessian (not the Gauss-Newton/NTK
approximation) — it therefore also sees negative curvature (saddles).

Two providers:
- `loss_hessian_at_epoch` &rarr; `NTKStats` of shape `(nreplicas, P, P)`
- `loss_hessian_eigenvalues_at_epoch` &rarr; `NTKStats` of shape `(nreplicas, P)` (descending)

Building the full Hessian costs ~$P$ JVPs per replica, so run it at a single epoch
(don't collect it over the whole trajectory).

In [1]:
%matplotlib inline
import numpy as np
from IPython.display import display

from ntkpdf.api import API
from ntkpdf.plotting.style import make_figure

import logging
logging.basicConfig(level=logging.INFO)

Applying NTKPDF plotting style...
Using Keras backend


In [2]:
# The data/theory context the FK chain needs, all read from the fit (see h_val_grid.ipynb).
common_dict = dict(
    dataset_inputs={"from_": "fit"},
    use_cuts="fromfit",
    theory={"from_": "fit"},
    theoryid={"from_": "theory"},
)
fit = "260527-ac-01-ntk-sgd"
epoch = 500

## Hessian and its eigenvalues

`training=True` builds the training-loss Hessian (`False` &rarr; validation loss).

In [ ]:
H = API.loss_hessian_at_epoch(fit=fit, epoch=epoch, training=True, **common_dict)
eigs = API.loss_hessian_eigenvalues_at_epoch(fit=fit, epoch=epoch, training=True, **common_dict)

nrep, P, _ = H.data.shape
print(f"Hessian: {H.data.shape}  (nreplicas, P, P);  eigenvalues: {eigs.data.shape}")
# Symmetric, and not positive semi-definite away from a minimum -> negative eigenvalues.
for r in range(nrep):
    ev = eigs.data[r]
    print(f"  replica {r+1}: eig in [{ev.min():.3g}, {ev.max():.3g}], "
          f"{int((ev < -1e-8).sum())} negative of {P}")

## Eigenvalue spectrum

Signed `symlog` y-axis so the (few) negative eigenvalues are visible alongside the
large positive bulk.

In [ ]:
fig, ax = make_figure()
for r in range(nrep):
    ax.plot(np.arange(1, P + 1), eigs.data[r], marker=".", ls="none", label=f"replica {r+1}")
ax.axhline(0.0, color="k", lw=0.6)
ax.set_yscale("symlog", linthresh=1e-3)
ax.set_xlabel(r"$\rm eigenvalue\ rank$")
ax.set_ylabel(r"$\rm Hessian\ eigenvalue$")
ax.set_title(rf"$\rm Loss\ Hessian\ spectrum$ ({fit}, epoch {epoch})")
ax.legend()
display(fig)

## Choosing the model specification

Pass `name=` to select which network the Hessian is taken with respect to — the same
selectors used elsewhere in the app (`produce_kwargs`): `"model"` (full, the default),
`"no_prefactors"` (neutralise the small/large-x prefactor exponents), `"nn"`,
`"no_smr"`. (For a layer-restricted Hessian, the same `grad_layers` mechanism as the
NTK applies via `ntkpdf.config.layer_kwargs`.)

In [ ]:
eigs_nopre = API.loss_hessian_eigenvalues_at_epoch(
    fit=fit, epoch=epoch, training=True, name="no_prefactors", **common_dict
)
print("no_prefactors eigenvalues:", eigs_nopre.data.shape)
for r in range(nrep):
    print(f"  replica {r+1}: largest eig = {eigs_nopre.data[r].max():.3g}")